# Notebook 4–5 (Integrado, v2) — Alinhamento, Retornos e Saneamento Robusto

Pipeline único, na ordem correta de execução:
**preços alinhados → painel → retornos → winsorização robusta.**

Revisão v2 (após parecer técnico externo), com duas melhorias:
1. **Escala robusta estimada sobre retornos não-nulos.** A mediana e o MAD passam a ser
   calculados apenas sobre os dias em que o ativo efetivamente negociou ($r\neq 0$). Isso
   corrige a **sobre-winsorização de papéis ilíquidos**, em que a alta proporção de dias de
   retorno zero comprimia o MAD e estreitava demais as cercas (e.g., SANB4 caía de ~11% para
   ~4% de observações winsorizadas; ativos líquidos como PETR4 praticamente não se alteram).
2. **Desfecho da verificação de eventos corporativos** (§7.3) documentado para os ativos
   sinalizados, atendendo à cobrança de registrar o resultado da checagem.

## Decisão metodológica central (inalterada): anomalias no domínio dos retornos

Em vez de excluir ativos por **nível de preço** (a antiga Etapa VI, discricionária e exposta a
viés de sobrevivência), a detecção de anomalias ocorre sobre os **retornos** (Campbell, Lo &
MacKinlay, 1997), via winsorização robusta por **Z-score modificado de Iglewicz & Hoaglin
(1993)**, $|M_i|>3{,}5$, uniforme a todos os ativos. O benchmark e a taxa livre de risco não
são tratados (seus extremos são reais). A exclusão por preço fica em toggle desligado.

## 1. Configuração — caminhos e parâmetros (limiares fixados na literatura)

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

parent_path = Path.cwd().parent.parent
INPUT_DIR_MATRIZ_PRECOS = parent_path / "data" / "Matriz_Precos"
INPUT_DIR_IBOV  = parent_path / "data" / "Ibov" / "Ibov_Diario"
INPUT_DIR_CDI   = parent_path / "data" / "CDI"
INPUT_DIR_SELIC = parent_path / "data" / "Selic"
OUTPUT_DIR      = parent_path / "data" / "Tickers"
DIR_PAINEL      = parent_path / "data" /  "Painel_Dados"
DIR_RETORNOS    = parent_path / "data" /  "Retornos"
for d in (DIR_PAINEL, DIR_RETORNOS):
    d.mkdir(parents=True, exist_ok=True)

EXCLUIR_PRECO_CORROMPIDO = False   # default: anomalias tratadas no domínio dos retornos
LIMIAR_PRECO_MAX         = 1000.0  # usado apenas se o toggle acima for True

K_MAD             = 3.5     # Iglewicz & Hoaglin (1993)
C_MAD             = 0.6745  # = Phi^{-1}(0,75)
MAD_SOBRE_NAO_NULOS = True  # v2: estima mediana/MAD apenas sobre retornos != 0
LIMITE_IMPOSSIVEL = 1.0     # |R| > 100% a.d. (flag de auditoria)
TRADING_DAYS      = 252

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
print(f"[OK] Exclusão por preço: {'LIGADA' if EXCLUIR_PRECO_CORROMPIDO else 'DESLIGADA'}")
print(f"[OK] MAD sobre retornos não-nulos: {MAD_SOBRE_NAO_NULOS}")
print(f"[OK] Winsorização MAD (Iglewicz-Hoaglin): k = {K_MAD}")

[OK] Exclusão por preço: DESLIGADA
[OK] MAD sobre retornos não-nulos: True
[OK] Winsorização MAD (Iglewicz-Hoaglin): k = 3.5


## 2. Carregamento dos insumos (131 ativos + IBOV + CDI + SELIC)

In [2]:
def _carregar_csv(path, date_col):
    if not path.exists():
        raise FileNotFoundError(f"{path.name} não encontrado em {path.parent}.")
    return pd.read_csv(path, parse_dates=[date_col]).set_index(date_col).sort_index()

precos = _carregar_csv(INPUT_DIR_MATRIZ_PRECOS / "Matriz_precos_sanitizada.csv", "Data")
print(f"Matriz de preços: {precos.shape[1]} ativos × {precos.shape[0]} pregões "
      f"({precos.index.min().date()} → {precos.index.max().date()})")

ibov_close   = _carregar_csv(INPUT_DIR_IBOV / "ibov_diario_2010_2026.csv", "data")["ibov_close"].rename("IBOV_close")
cdi_diario   = _carregar_csv(INPUT_DIR_CDI / "cdi_diario_bcb_2010_atual.csv", "data")["cdi_diario"].rename("CDI_diario")
selic_diario = _carregar_csv(INPUT_DIR_SELIC / "selic_diario_bcb_2010_atual.csv", "data")["selic_diario"].rename("SELIC_diario")
print(f"IBOVESPA: {len(ibov_close):,} | CDI: {len(cdi_diario):,} | SELIC: {len(selic_diario):,} obs")

_excl = INPUT_DIR_MATRIZ_PRECOS / "tickers_excluidos.csv"
excluidos_prev = (pd.read_csv(_excl) if _excl.exists()
                  else pd.DataFrame(columns=["ticker","presenca_pct","primeiro_pregao_real","motivo"]))
print(f"Registro de exclusões anterior: {len(excluidos_prev)} tickers")

Matriz de preços: 131 ativos × 3967 pregões (2010-01-04 → 2025-12-30)
IBOVESPA: 3,967 | CDI: 4,019 | SELIC: 4,019 obs
Registro de exclusões anterior: 365 tickers


## 3. Etapa VI — diagnóstico de integridade (sem exclusão por padrão)

In [3]:
rets_diag = precos.pct_change()
integridade = pd.DataFrame({
    "preco_inicial": precos.iloc[0], "preco_final": precos.iloc[-1],
    "preco_max": precos.max(), "preco_min": precos.min(),
    "amplitude": precos.max() / precos.min(),
    "saltos_>50pct": (rets_diag.abs() > 0.50).sum(),
})
mask_corrompido = integridade["preco_max"] > LIMIAR_PRECO_MAX
print(f"=== Diagnóstico: preço máximo > R$ {LIMIAR_PRECO_MAX:,.0f} ({int(mask_corrompido.sum())}) ===")
print(integridade[mask_corrompido].sort_values("preco_max", ascending=False)
      [["preco_inicial","preco_max","amplitude","saltos_>50pct"]].to_string())

if EXCLUIR_PRECO_CORROMPIDO:
    precos_VI = precos.drop(columns=integridade.index[mask_corrompido])
    novos = pd.DataFrame({"ticker": integridade.index[mask_corrompido], "presenca_pct": np.nan,
                          "primeiro_pregao_real": np.nan,
                          "motivo": f"preco max > R$ {LIMIAR_PRECO_MAX:.0f} (Etapa VI)"})
    excluidos_prev = pd.concat([excluidos_prev, novos], ignore_index=True)
    print(f"\n[Etapa VI LIGADA] {precos.shape[1]} → {precos_VI.shape[1]} ativos.")
else:
    precos_VI = precos
    print(f"\n[Etapa VI DESLIGADA] Mantidos os {precos_VI.shape[1]} ativos.")

=== Diagnóstico: preço máximo > R$ 1,000 (9) ===
             preco_inicial             preco_max             amplitude  saltos_>50pct
PDGR3 8,661,504,016.799999 12,097,253,755.000000 14,752,748,481.707317             12
GFSA3        12,025.947461         12,927.532550          2,756.403529              1
AMER3         3,863.522415         12,289.634644          3,926.400845              3
OIBR3        11,703.943587         11,955.191035        239,103.820700              4
VIVR3        10,547.946999         11,885.438837         40,173.867963              5
OIBR4         6,521.765601          6,895.491686          4,364.235245              1
LUPA3         4,777.215667          5,206.833326          6,590.928261              7
NEXP3         1,722.570661          2,668.824770            923.468778              1
RCSL4         1,298.728113          1,650.977483         10,005.924140              5

[Etapa VI DESLIGADA] Mantidos os 131 ativos.


## 4. Alinhamento temporal por interseção de calendário

In [4]:
calendario = (precos_VI.index.intersection(ibov_close.index)
               .intersection(cdi_diario.index).intersection(selic_diario.index).sort_values())
print(f"Calendário de interseção: {len(calendario):,} pregões "
      f"({calendario.min().date()} → {calendario.max().date()})")
gaps = pd.Series(calendario).diff().dt.days
print("\nMaiores intervalos entre pregões (feriados prolongados esperados):")
print(pd.DataFrame({"data": calendario, "gap_dias": gaps.values}).nlargest(5,"gap_dias").to_string(index=False))

Calendário de interseção: 3,967 pregões (2010-01-04 → 2025-12-30)

Maiores intervalos entre pregões (feriados prolongados esperados):
      data  gap_dias
2010-02-17  5.000000
2011-03-09  5.000000
2011-04-25  5.000000
2012-02-22  5.000000
2012-12-26  5.000000


## 5. Painel consolidado e exportação

In [5]:
acoes_alinhado = precos_VI.reindex(calendario)
acoes_alinhado.columns = [f"ACAO_{c}" for c in acoes_alinhado.columns]
painel = pd.concat([ibov_close.reindex(calendario), cdi_diario.reindex(calendario),
                    selic_diario.reindex(calendario), acoes_alinhado], axis=1)
painel.index.name = "data"
n_nan = int(painel.isna().sum().sum())
print(f"Painel: {painel.shape[0]:,} pregões × {painel.shape[1]} colunas | NaN: {n_nan}")
assert n_nan == 0, "Há NaN no painel — investigar."
painel.to_parquet(DIR_PAINEL / "painel_alinhado.parquet", engine="pyarrow")
painel.to_csv(DIR_PAINEL / "painel_alinhado.csv", date_format="%Y-%m-%d", float_format="%.8f")
print("[OK] Painel exportado.")

Painel: 3,967 pregões × 134 colunas | NaN: 0
[OK] Painel exportado.


## 6. Matriz de retornos (simples e logarítmica)

In [6]:
cols_acoes = [c for c in painel.columns if c.startswith("ACAO_")]
precos_acoes = painel[cols_acoes]
ret_simples = precos_acoes.pct_change().iloc[1:]
ret_log     = np.log(precos_acoes).diff().iloc[1:]
ibov_ret_simples = painel["IBOV_close"].pct_change().iloc[1:]
ibov_ret_log     = np.log(painel["IBOV_close"]).diff().iloc[1:]
rf = painel[["CDI_diario","SELIC_diario"]].reindex(ret_simples.index)
print(f"Retornos: {ret_simples.shape[0]} pregões × {ret_simples.shape[1]} ativos")
assert np.allclose(ret_log.values, np.log1p(ret_simples.values), atol=1e-10)
print("[OK] Identidade log ≡ ln(1+simples) verificada (brutos).")

Retornos: 3966 pregões × 131 ativos
[OK] Identidade log ≡ ln(1+simples) verificada (brutos).


## 7. Saneamento robusto — winsorização por Z-score modificado (MAD, k=3,5)

**v2:** a escala robusta (mediana e MAD) é estimada sobre os retornos **não-nulos** de cada
ativo, evitando que dias de preço travado (iliquidez/ffill) comprimam artificialmente as cercas.
O log registra, por ativo, o número de winsorizações com e sem essa correção, para auditoria.
Aplicada **apenas às ações**; benchmark e $r_f$ ficam intactos.

In [7]:
# --- 7.1 Auditoria: retornos economicamente impossíveis ---
impossiveis = (ret_simples.abs() > LIMITE_IMPOSSIVEL).sum()
flag = impossiveis[impossiveis > 0].sort_values(ascending=False)
audit = pd.DataFrame({"dias_|R|>100%": flag,
                      "maior_|R|_%": (ret_simples[flag.index].abs().max()*100).round(0),
                      "preco_mediano_R$": painel[flag.index].median().round(4)})
print("=== Ativos com |R|>100% (ver desfecho em §7.3) ===")
print(audit.to_string() if len(audit) else "Nenhum.")

# --- 7.2 Winsorização robusta ---
def cercas_mad(serie, k=K_MAD, c=C_MAD, nao_nulos=MAD_SOBRE_NAO_NULOS):
    x = serie.astype(float).values
    base = x[x != 0] if nao_nulos else x
    if base.size == 0:
        return None, None
    med = np.median(base); mad = np.median(np.abs(base - med))
    if mad == 0:
        return None, None
    return med - k*mad/c, med + k*mad/c

ret_log_san = ret_log.copy()
relatorio = []
for a in cols_acoes:
    lo, hi = cercas_mad(ret_log[a])                       # v2 (sobre não-nulos)
    lo0, hi0 = cercas_mad(ret_log[a], nao_nulos=False)    # referência (com zeros)
    n0 = 0 if lo0 is None else int(((ret_log[a] < lo0) | (ret_log[a] > hi0)).sum())
    if lo is None:
        relatorio.append({"ativo": a, "n_winsor_v2": 0, "n_winsor_com_zeros": n0, "mad_zero": True})
        continue
    n = int(((ret_log[a] < lo) | (ret_log[a] > hi)).sum())
    ret_log_san[a] = ret_log[a].clip(lower=lo, upper=hi)
    relatorio.append({"ativo": a, "n_winsor_v2": n, "n_winsor_com_zeros": n0, "mad_zero": False})

ret_simples_san = np.expm1(ret_log_san)
rel = pd.DataFrame(relatorio).set_index("ativo")
n_v2, n_old = int(rel["n_winsor_v2"].sum()), int(rel["n_winsor_com_zeros"].sum())
print(f"\n[+] Winsorizadas (v2, sobre não-nulos): {n_v2} "
      f"({n_v2/(len(cols_acoes)*len(ret_log))*100:.3f}%)")
print(f"    Referência (MAD com zeros):         {n_old} "
      f"({n_old/(len(cols_acoes)*len(ret_log))*100:.3f}%)  -> redução de {n_old-n_v2} obs")
print("\n=== Ativos antes super-winsorizados (maior queda v2 vs. com-zeros) ===")
rel["reducao"] = rel["n_winsor_com_zeros"] - rel["n_winsor_v2"]
print(rel.sort_values("reducao", ascending=False).head(8)
        [["n_winsor_com_zeros","n_winsor_v2","reducao"]].to_string())

=== Ativos com |R|>100% (ver desfecho em §7.3) ===
             dias_|R|>100%  maior_|R|_%  preco_mediano_R$
ACAO_FICT3               2   142.000000          0.627100
ACAO_GOLL54              2 1,847.000000          0.148000
ACAO_LUPA3               2   148.000000          3.528900
ACAO_TELB4               2   223.000000         22.509600
ACAO_AMER3               1   180.000000      1,616.798000

[+] Winsorizadas (v2, sobre não-nulos): 10258 (1.974%)
    Referência (MAD com zeros):         13673 (2.632%)  -> redução de 3415 obs

=== Ativos antes super-winsorizados (maior queda v2 vs. com-zeros) ===
            n_winsor_com_zeros  n_winsor_v2  reducao
ativo                                               
ACAO_SANB4                 436          159      277
ACAO_SANB3                 305           52      253
ACAO_RCSL4                 500          304      196
ACAO_FICT3                 433          253      180
ACAO_VIVR3                 339          215      124
ACAO_BEES3             

### 7.3 Desfecho da verificação de eventos corporativos

Resultado da checagem dos ativos sinalizados (todos confirmados como **tickers reais** da B3;
nenhum é sentinela/teste). A natureza dos retornos extremos e a decisão de tratamento ficam
registradas para o apêndice e para arguição.

In [8]:
verificacao = pd.DataFrame([
 {"ticker":"FICT3","empresa":"Fictor Alimentos (ex-ATOM3)",
  "evento":"IPO reverso (troca de controle/ticker) + RJ da holding (2026); penny stock < R$1",
  "natureza_extremo":"Distress real + mudanca de base acionaria","decisao":"Manter (ativo real)"},
 {"ticker":"GOLL54","empresa":"Gol Linhas Aereas",
  "evento":"Reestruturacao/distress (2024-25); salto +1847% = provavel grupamento nao-ajustado",
  "natureza_extremo":"Artefato de evento (denominador ~centavo)",
  "decisao":"Manter; ideal ajustar preco a montante pelo grupamento"},
 {"ticker":"LUPA3","empresa":"Lupatech",
  "evento":"RJ 2015-2023 c/ conversao de ~85% da divida em acoes (forte diluicao); nova RE 2026",
  "natureza_extremo":"Diluicao/distress real","decisao":"Manter"},
 {"ticker":"TELB4","empresa":"Telebras (PN)",
  "evento":"Multiplos splits/grupamentos no historico; liquidez muito baixa",
  "natureza_extremo":"Artefato de split/grupamento + iliquidez","decisao":"Manter; conferir datas"},
 {"ticker":"AMER3","empresa":"Americanas",
  "evento":"Fraude contabil 11/01/2023 -> RJ 19/01/2023; grupamento 100:1 em ago/2024",
  "natureza_extremo":"Crash real (fraude) + nivel ajustado inflado pelo inplit 100:1",
  "decisao":"Manter (anti-sobrevivencia)"},
]).set_index("ticker")

print(verificacao[["empresa","natureza_extremo","decisao"]].to_string())
verificacao.to_csv(DIR_RETORNOS / "verificacao_eventos_corporativos.csv")
print("\n[OK] Desfecho salvo em verificacao_eventos_corporativos.csv")

                            empresa                                                natureza_extremo                                                 decisao
ticker                                                                                                                                                     
FICT3   Fictor Alimentos (ex-ATOM3)                       Distress real + mudanca de base acionaria                                     Manter (ativo real)
GOLL54            Gol Linhas Aereas                       Artefato de evento (denominador ~centavo)  Manter; ideal ajustar preco a montante pelo grupamento
LUPA3                      Lupatech                                          Diluicao/distress real                                                  Manter
TELB4                 Telebras (PN)                        Artefato de split/grupamento + iliquidez                                  Manter; conferir datas
AMER3                    Americanas  Crash real (fraude) + nivel

## 8. Testes de integridade finais

In [9]:
print("[+] Verificando integridade...\n")
assert ret_log_san.index.equals(ret_log.index) and ret_log_san.shape == ret_simples_san.shape == ret_log.shape
print("      [OK] Índice e dimensões preservados")
assert ret_log_san.isna().sum().sum()==0 and ret_simples_san.isna().sum().sum()==0
assert np.isfinite(ret_log_san.values).all() and np.isfinite(ret_simples_san.values).all()
print("      [OK] Sem NaN ou infinitos")
assert np.allclose(ret_log_san.values, np.log1p(ret_simples_san.values), atol=1e-10)
print("      [OK] Identidade log ≡ ln(1+simples) preservada")
mask_validos = ~rel["mad_zero"].values
resid = int((ret_simples_san.loc[:, mask_validos].abs() > LIMITE_IMPOSSIVEL).sum().sum())
print(f"      [OK] |R|>100% remanescentes (MAD>0): {resid}")
print("\n[OK] Matrizes saneadas aprovadas.")

[+] Verificando integridade...

      [OK] Índice e dimensões preservados
      [OK] Sem NaN ou infinitos
      [OK] Identidade log ≡ ln(1+simples) preservada
      [OK] |R|>100% remanescentes (MAD>0): 0

[OK] Matrizes saneadas aprovadas.


## 9. Exportação dos artefatos finais

In [10]:
def _salvar(df, nome, ddir=DIR_RETORNOS):
    df.to_csv(ddir / f"{nome}.csv", index=True, date_format="%Y-%m-%d", float_format="%.8f")
    try:
        df.to_parquet(ddir / f"{nome}.parquet", engine="pyarrow"); print(f"  {nome}.csv + .parquet")
    except Exception as e: print(f"  {nome}.csv  [parquet: {e}]")

print("[+] Gravando em:", DIR_RETORNOS, "\n")
_salvar(ret_simples, "retornos_simples"); _salvar(ret_log, "retornos_log")
_salvar(ret_simples_san, "retornos_simples_saneado"); _salvar(ret_log_san, "retornos_log_saneado")
_salvar(pd.DataFrame({"ibov_ret_simples": ibov_ret_simples, "ibov_ret_log": ibov_ret_log}), "ibov_retornos")
_salvar(rf.rename(columns=str.lower), "rf_diario")
rel.to_csv(DIR_RETORNOS / "log_winsorizacao.csv", float_format="%.8f")
if len(audit): audit.to_csv(DIR_RETORNOS / "flag_integridade_eventos.csv", float_format="%.4f")
pd.Series([c.replace("ACAO_","") for c in cols_acoes], name="ticker").to_csv(OUTPUT_DIR / "tickers_finais.csv", index=False)
excluidos_prev.to_csv(OUTPUT_DIR / "tickers_excluidos.csv", index=False)

print("\n" + "="*62)
print("PIPELINE INTEGRADO (v2) CONCLUÍDO")
print("="*62)
print(f"Ativos no painel: {len(cols_acoes)}  |  Pregões de retorno: {ret_simples_san.shape[0]}")
print(f"Winsorizadas (v2): {n_v2} ({n_v2/(len(cols_acoes)*len(ret_log))*100:.3f}%) "
      f"| antes (com zeros): {n_old}")
print("Insumo dos otimizadores: retornos_*_saneado (NB 6+).")
print("="*62)

[+] Gravando em: c:\VSCodeWorkspace\TCC_Final\data\Retornos 

  retornos_simples.csv + .parquet
  retornos_log.csv + .parquet
  retornos_simples_saneado.csv + .parquet
  retornos_log_saneado.csv + .parquet
  ibov_retornos.csv + .parquet
  rf_diario.csv + .parquet

PIPELINE INTEGRADO (v2) CONCLUÍDO
Ativos no painel: 131  |  Pregões de retorno: 3966
Winsorizadas (v2): 10258 (1.974%) | antes (com zeros): 13673
Insumo dos otimizadores: retornos_*_saneado (NB 6+).


## 10. Apêndice para o TCC — texto de redação

> *"O tratamento de anomalias foi conduzido no domínio dos retornos, e não por exclusão de
> ativos com base no nível de preço, evitando a discricionariedade na seleção de tickers e o
> viés de sobrevivência (Brown, Goetzmann e Ross, 1992). Sobre os log-retornos de cada ação
> aplicou-se o Z-score modificado de Iglewicz e Hoaglin (1993), winsorizando-se as observações
> com $|M_i|>3{,}5$. A mediana e o desvio absoluto mediano que definem as cercas foram
> estimados sobre os retornos não-nulos de cada ativo, evitando que dias de preço travado em
> papéis de baixa liquidez comprimissem artificialmente as cercas. O índice de mercado e a taxa
> livre de risco, cujos extremos são economicamente legítimos, foram preservados. Os ativos com
> retorno diário economicamente impossível ($|R|>100\%$) foram verificados contra seus eventos
> corporativos: todos correspondem a companhias reais cujos extremos decorrem de grupamentos,
> diluições ou episódios de distress (Fictor/FICT3, Gol/GOLL54, Lupatech/LUPA3, Telebras/TELB4,
> Americanas/AMER3), tendo sido mantidos na amostra com tratamento no domínio dos retornos."*

**Artefatos em `data/`:** `Painel_Dados/painel_alinhado.*`; `Retornos/retornos_{simples,log}.*`
e `..._saneado.*`; `ibov_retornos.*`; `rf_diario.*`; `log_winsorizacao.csv`;
`flag_integridade_eventos.csv`; `verificacao_eventos_corporativos.csv`; e `tickers_finais.csv`.